# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset ["Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and display a short description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata
dataset = mlc.Dataset(croissant_url)
print(f"Title: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields they contain. All entities are referenced by their `@id` according to the Croissant schema.

We will list each record set, and for each, we enumerate their fields (columns) using the `@id` of each component.

In [ ]:
# List all available record sets and their details from the metadata
for record_set in dataset.metadata.record_sets:
    print(f"Record Set: {record_set.name}")
    print(f"  @id: {record_set.id}")
    fields = getattr(record_set, 'fields', [])
    print("  Fields:")
    for field in fields:
        print(f"    - {field.name} @id: {field.id} [dataType: {field.data_type if hasattr(field, 'data_type') else 'N/A'}]")
    print()

## 3. Data Extraction
We load data from each record set into Pandas DataFrames for further analysis. For each, we use its `@id` and display the resulting columns.

In [ ]:
# Collect all record set @ids
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Record set @ids: {record_sets_ids}")

dataframes = {}

# Try loading 5 records (if any) for each record set @id
for record_set_id in record_sets_ids:
    print(f"Loading records from Record Set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for Record Set {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Let's select a record set with data, choose a numeric field, filter the data, normalize it, and group or aggregate as an example. All field references are by their `@id` as in the schema.

Adjust the field names below to the actual `@id` of a numeric and group field from the available columns listed previously.

In [ ]:
# Pick one record set with data (update as appropriate)
if len(dataframes) > 0:
    chosen_record_set_id = list(dataframes.keys())[0]
    print(f"Using record set: {chosen_record_set_id}")
    df = dataframes[chosen_record_set_id]

    # Try to identify the first numeric field (float or int; update by inspecting df.head())
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found to perform EDA.")
    else:
        threshold = df[numeric_field_id].mean()  # Use mean as an example threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered rows where '{numeric_field_id}' > {threshold:.2f} (mean):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by the first non-numeric field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Mean of '{numeric_field_id}' grouped by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No non-numeric group field found.")
else:
    print("No record sets with data to perform EDA.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field and how it varies by a group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset, explored its available record sets and fields using their `@id`s, extracted data, and performed basic exploratory analysis and visualizations. 

- All referencing to schema elements was done via `@id` for reproducibility.
- You can adapt field choices to your analysis by listing the available IDs, inspecting fields, and applying advanced analytics as needed.

**Next steps:** Try filtering by other fields, joining across record sets, or repeating these operations on other datasets described by Croissant schemas.